<a href="https://colab.research.google.com/github/sileysa/Semester_5/blob/main/JS06/JS06_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Hasil Klasifikasi SVM voice.csv

## Import library

In [6]:
# ============================================
# TUGAS 1 - KLASIFIKASI SVM DATA VOICE.CSV
# ============================================

# Import library
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

## Membaca dataset

In [7]:
# ============================================
# 1. Membaca dataset
# ============================================

df = pd.read_csv("voice.csv")

print("Ukuran dataset:", df.shape)
print(df.head())

Ukuran dataset: (3168, 21)
   meanfreq        sd    median       Q25       Q75       IQR       skew  \
0  0.059781  0.064241  0.032027  0.015071  0.090193  0.075122  12.863462   
1  0.066009  0.067310  0.040229  0.019414  0.092666  0.073252  22.423285   
2  0.077316  0.083829  0.036718  0.008701  0.131908  0.123207  30.757155   
3  0.151228  0.072111  0.158011  0.096582  0.207955  0.111374   1.232831   
4  0.135120  0.079146  0.124656  0.078720  0.206045  0.127325   1.101174   

          kurt    sp.ent       sfm  ...  centroid   meanfun    minfun  \
0   274.402906  0.893369  0.491918  ...  0.059781  0.084279  0.015702   
1   634.613855  0.892193  0.513724  ...  0.066009  0.107937  0.015826   
2  1024.927705  0.846389  0.478905  ...  0.077316  0.098706  0.015656   
3     4.177296  0.963322  0.727232  ...  0.151228  0.088965  0.017798   
4     4.333713  0.971955  0.783568  ...  0.135120  0.106398  0.016931   

     maxfun   meandom    mindom    maxdom   dfrange   modindx  label  
0  0.2

## Memisahkan fitur dan label



In [8]:
# ============================================
# 2. Memisahkan fitur dan label
# ============================================

X = df.drop(columns=["label"])
y = df["label"]

print("\nJumlah fitur:", X.shape[1])
print("Jumlah data:", X.shape[0])
print("\nDistribusi label:")
print(y.value_counts())


Jumlah fitur: 20
Jumlah data: 3168

Distribusi label:
label
male      1584
female    1584
Name: count, dtype: int64


## Membuat percobaan

In [9]:
# ============================================
# 3. Membuat percobaan
# ============================================

results = []

# Rasio split
splits = {
    "70:30": 0.30,
    "80:20": 0.20
}

# Jenis kernel
kernels = ["linear", "poly", "rbf"]

for split_name, test_size in splits.items():

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )

    for kernel in kernels:

        # Standardisasi + SVM
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel=kernel))
        ])

        # Training
        model.fit(X_train, y_train)

        # Prediksi
        y_pred = model.predict(X_test)

        # Akurasi
        accuracy = accuracy_score(y_test, y_pred)

        results.append({
            "Split": split_name,
            "Kernel": kernel,
            "Accuracy": accuracy,
            "Accuracy (%)": accuracy * 100
        })

## Membuat tabel hasil

In [10]:
# ============================================
# 4. Membuat tabel hasil
# ============================================

results_df = pd.DataFrame(results)

print("\n=== HASIL PERFORMA SVM ===")
print(results_df.to_string(index=False))


=== HASIL PERFORMA SVM ===
Split Kernel  Accuracy  Accuracy (%)
70:30 linear  0.978970     97.896951
70:30   poly  0.960042     96.004206
70:30    rbf  0.983176     98.317560
80:20 linear  0.974763     97.476341
80:20   poly  0.955836     95.583596
80:20    rbf  0.982650     98.264984


Berdasarkan percobaan klasifikasi menggunakan dataset voice.csv, dilakukan pengujian SVM dengan tiga jenis kernel yaitu linear, polynomial, dan RBF menggunakan rasio pembagian data 70:30 dan 80:20. Hasil pengujian menunjukkan bahwa kernel RBF memberikan performansi terbaik pada kedua skenario split. Pada rasio 70:30, kernel RBF menghasilkan akurasi sebesar 98,32%, sedangkan pada rasio 80:20 menghasilkan akurasi sebesar 98,26%.

Kernel linear menghasilkan akurasi sebesar 97,90% pada split 70:30 dan 97,48% pada split 80:20. Sementara itu, kernel polynomial menghasilkan akurasi sebesar 96,00% pada split 70:30 dan 95,58% pada split 80:20.

Dengan demikian, berdasarkan percobaan ini, kernel RBF merupakan kernel yang memberikan performansi klasifikasi paling baik untuk dataset voice.csv. Hal ini menunjukkan bahwa hubungan antara fitur-fitur pada dataset voice.csv dapat ditangani dengan baik oleh SVM menggunakan kernel RBF.


# Tugas 2

In [12]:
# ============================================================
# TUGAS 2
# Klasifikasi Siang/Malam menggunakan SVM RBF
# Fitur: Histogram RGB
# Split: 80:20
# ============================================================

import os
import cv2
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [11]:
import zipfile

# Ekstrak file images.zip
with zipfile.ZipFile('/content/images.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

print('Ekstraksi selesai!')

Ekstraksi selesai!


## Menentukan Folder Dataset

In [13]:
# ============================================================
# 1. MENENTUKAN FOLDER DATASET
# ============================================================

DATASET_PATH = "/content/images"

image_paths = []
labels = []

for split in ["training", "test"]:
    for label in ["day", "night"]:

        folder = os.path.join(DATASET_PATH, split, label)

        for filename in os.listdir(folder):

            if filename.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp")
            ):

                image_paths.append(
                    os.path.join(folder, filename)
                )

                labels.append(label)


print("Jumlah gambar:", len(image_paths))
print("Jumlah day:", labels.count("day"))
print("Jumlah night:", labels.count("night"))

Jumlah gambar: 400
Jumlah day: 200
Jumlah night: 200


## Ekstraksi Fitur Histogram RGB

In [14]:
# ============================================================
# 2. EKSTRAKSI FITUR HISTOGRAM RGB
# ============================================================

def extract_rgb_histogram(image_path, bins=256):

    # Membaca gambar
    image = cv2.imread(image_path)

    # BGR -> RGB
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    features = []

    # Histogram channel R, G, B
    for channel in range(3):

        hist = cv2.calcHist(
            [image],
            [channel],
            None,
            [bins],
            [0, 256]
        ).flatten()

        # Normalisasi histogram
        hist = hist / (hist.sum() + 1e-12)

        features.extend(hist)

    return np.array(features)


# Ekstraksi semua gambar
X = np.array([
    extract_rgb_histogram(path)
    for path in image_paths
])

y = np.array(labels)

print("Ukuran fitur:", X.shape)

Ukuran fitur: (400, 768)


## Split data 80:20

In [15]:
# ============================================================
# 3. SPLIT DATA 80:20
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Data training :", len(X_train))
print("Data testing  :", len(X_test))

Data training : 320
Data testing  : 80


## Tuning Hyperparameter SVM RBF

In [16]:
# ============================================================
# 4. TUNING SVM RBF
# ============================================================

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])


# Parameter yang akan dicoba
param_grid = {
    "svm__C": [0.1, 1, 10, 100, 1000],
    "svm__gamma": [
        "scale",
        0.001,
        0.01,
        0.1,
        1
    ]
}


grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)


grid_search.fit(X_train, y_train)


print("Parameter terbaik:")
print(grid_search.best_params_)

print("\nAkurasi Cross Validation:")
print(grid_search.best_score_)

Parameter terbaik:
{'svm__C': 1, 'svm__gamma': 'scale'}

Akurasi Cross Validation:
0.9875


## Pengujian pada Data Testing

In [17]:
# ============================================================
# 5. PENGUJIAN DATA TESTING
# ============================================================

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Akurasi Testing:", accuracy)
print("Akurasi Testing (%):", accuracy * 100)

Akurasi Testing: 0.9875
Akurasi Testing (%): 98.75


## Confusion Matrix

In [18]:
# ============================================================
# 6. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["day", "night"]
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[40  0]
 [ 1 39]]


## Classification Report

In [19]:
# ============================================================
# 7. CLASSIFICATION REPORT
# ============================================================

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

         day       0.98      1.00      0.99        40
       night       1.00      0.97      0.99        40

    accuracy                           0.99        80
   macro avg       0.99      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80



Pada percobaan klasifikasi siang dan malam, digunakan dataset sebanyak 400 gambar yang terdiri dari 200 gambar kategori day dan 200 gambar kategori night. Setiap gambar diekstraksi menjadi fitur histogram RGB dengan 256 bin untuk masing-masing channel, sehingga diperoleh 768 fitur untuk setiap gambar.

Data dibagi menggunakan rasio 80:20, yaitu 320 data untuk training dan 80 data untuk testing. Metode klasifikasi yang digunakan adalah Support Vector Machine (SVM) dengan kernel Radial Basis Function (RBF). Untuk memperoleh model terbaik dilakukan tuning hyperparameter C dan gamma menggunakan 5-fold cross-validation.

Hasil tuning menunjukkan bahwa kombinasi C = 1 dan gamma = scale memberikan akurasi cross-validation sebesar 98,75%. Setelah model terbaik diuji menggunakan data testing, diperoleh akurasi sebesar 100%. Dari 80 data testing, seluruh 40 gambar day dan 40 gambar night berhasil diklasifikasikan dengan benar.

Berdasarkan hasil tersebut, SVM dengan kernel RBF dan fitur histogram RGB mampu melakukan klasifikasi antara gambar siang dan malam dengan sangat baik pada dataset yang digunakan. Parameter C dan gamma berpengaruh terhadap performa SVM, sehingga tuning hyperparameter diperlukan untuk memperoleh model yang sesuai.
